# COE305 Machine Learning Project
## Predicting University Global Rank

**Team Members:** Yağız Efe Gökçe, Kerem Özcan, Murat Emre Doğan

### 1. Introduction
This project aims to build predictive machine learning models to estimate a university's global ranking score and category based on objective institutional metrics such as quality of education, alumni employment, and publications.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, classification_report
import xgboost as xgb
import shap

%matplotlib inline

### 2. Data Loading and Cleaning

In [ ]:
def load_and_clean_data(filepath="data/cwurData.csv"):
    df = pd.read_csv(filepath)
    
    # Impute missing values
    if 'broad_impact' in df.columns:
        df['broad_impact'] = df['broad_impact'].fillna(df['broad_impact'].median())
    
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in numerical_cols:
        df[col] = df[col].fillna(df[col].median())
        
    df.drop_duplicates(inplace=True)
    return df

df = load_and_clean_data()
df.head()

### 3. Feature Engineering

In [ ]:
def categorize_rank(rank):
    if rank <= 100:
        return 0 # Elite
    elif rank <= 500:
        return 1 # High
    else:
        return 2 # Average
    
if 'world_rank' in df.columns:
    df['ranking_category'] = df['world_rank'].apply(categorize_rank)

# Encode Country
if 'country' in df.columns:
    le = LabelEncoder()
    df['country_encoded'] = le.fit_transform(df['country'].astype(str))

df[['world_rank', 'ranking_category', 'country', 'country_encoded']].head()

### 4. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['score'], kde=True, bins=30)
plt.title("Distribution of University Scores")
plt.xlabel("Score")
plt.show()

### 5. Model Training: Regression (Predicting Score)

In [ ]:
# Prepare Data for Regression
drop_cols = ['institution', 'year', 'country', 'ranking_category', 'score', 'world_rank']
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]
y = df['score']

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
}

print("--- Regression Results ---")
for name, model in reg_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name}: RMSE={rmse:.4f}, R2={r2:.4f}")

### 6. Model Training: Classification (Predicting Category)

In [ ]:
# Prepare Data for Classification
y_class = df['ranking_category']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class)

# Initialize all 6 models as per report
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}

print("--- Classification Results ---")
for name, model in clf_models.items():
    model.fit(X_train_c, y_train_c)
    y_pred = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, y_pred)
    f1 = f1_score(y_test_c, y_pred, average='weighted')
    print(f"{name}: Accuracy={acc:.4f}, F1 Score={f1:.4f}")

### 7. SHAP Analysis

In [ ]:
# Explain the XGBoost model using SHAP
best_model = clf_models['XGBoost']
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_scaled)

# Summary Bar Plot
plt.title("SHAP Feature Importance")
shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols, plot_type="bar")

# Beeswarm Plot
plt.title("SHAP Summary (Beeswarm)")
shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols)